In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

In [ ]:
colors_ordered = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
    "tab:purple",
    "tab:brown",
    "tab:pink",
]

internal_state_color = "tab:olive"


def make_plot(ax, data, data_range, linestyle='-', **kwargs):
    """
    Plots multiple columns of data with a consistent color cycle.
    
    Args:
        ax: The matplotlib axis to plot on.
        data: The 2D array/dataframe containing the values.
        data_range: The indices of the columns to plot.
        linestyle: Standard linestyle (solid, dashed, etc.)
        **kwargs: Additional styling like alpha, linewidth, or markers.
    """
    for i, j in enumerate(data_range):
        ax.plot(
            data[:, j], 
            linestyle=linestyle, 
            color=colors_ordered[i % len(colors_ordered)],
            **kwargs  # This catches alpha, linewidth, etc., from the call
        )

In [ ]:
freq = 740
data_version = "benchmark"

suffix_tct = f"{data_version}_freq{freq}.npy"

tct_data_in = np.load(f"results/tct_results_in_{suffix_tct}")
tct_data_out = np.load(f"results/tct_results_out_{suffix_tct}")
tct_data_internal = np.load(f"results/tct_results_internal_{suffix_tct}")

In [ ]:
method = "lstm"
version = "01"

suffix_sm = f"{data_version}_{method}_v{version}_freq{freq}.npy"

sm_results_in = np.load(f"results/sm_results_in_{suffix_sm}")
sm_results_out = np.load(f"results/sm_results_out_{suffix_sm}")
sm_results_internal = np.load(f"results/sm_results_internal_{suffix_sm}")

In [ ]:
show_predictions = True

method_legend_handles = {
    "spils_net": "SPILS-Net",
    "lstm": "LSTM",
}

num_points = tct_data_in.shape[1] // 3
if data_version == "benchmark":
    nodes_plot = 6
else:
    nodes_plot = 16

plt.rcParams.update({
    "font.family": "serif",   # Matches LaTeX default
    "font.size": 11,          # Standard paper font size
    "savefig.dpi": 300,       # Default save DPI
})

# 1. Change layout to 5 rows, 1 column. 
# 2. Use sharex=True so we only need one x-axis label at the very bottom.
if method == "spils_net":
    fig, axs = plt.subplots(5, 1, figsize=(10, 15), sharex=True)
else:
    fig, axs = plt.subplots(4, 1, figsize=(10, 12), sharex=True)

# Set y-labels. (Adding newlines keeps the text closer to the axis if it gets too wide)
axs[0].set_ylabel('Displacement x\n[mm]')
axs[1].set_ylabel('Displacement y\n[mm]')
axs[2].set_ylabel('Force x\n[N]')
axs[3].set_ylabel('Force y\n[N]')
if method == "spils_net":
    axs[4].set_ylabel('Internal State')

    # Only set the x-label on the bottom-most plot
    axs[4].set_xlabel("Time step")
else:
    axs[3].set_xlabel("Time step")

# Plot TRUE data: Make it thicker and slightly transparent (alpha=0.5) 
# This turns the true data into a visible "background" track.
# Note: This assumes your `make_plot` function accepts **kwargs like linewidth/alpha.
make_plot(axs[0], tct_data_in, range(0, num_points, nodes_plot), linewidth=3, alpha=0.5)
make_plot(axs[1], tct_data_in, range(1, num_points, nodes_plot), linewidth=3, alpha=0.5)
make_plot(axs[2], tct_data_out, range(0, num_points, nodes_plot), linewidth=3, alpha=0.5)
make_plot(axs[3], tct_data_out, range(1, num_points, nodes_plot), linewidth=3, alpha=0.5)
if method == "spils_net":
    axs[4].plot(tct_data_internal, linewidth=3, alpha=0.5, color=internal_state_color)

if show_predictions:
    # Plot PREDICTION data: Thinner, dashed, and fully opaque.
    # This allows the prediction to snap sharply over the semi-transparent true lines.
    make_plot(axs[0], sm_results_in, range(0, num_points, nodes_plot), linestyle='--', linewidth=1.5)
    make_plot(axs[1], sm_results_in, range(1, num_points, nodes_plot), linestyle='--', linewidth=1.5)
    make_plot(axs[2], sm_results_out, range(0, num_points, nodes_plot), linestyle='--', linewidth=1.5)
    make_plot(axs[3], sm_results_out, range(1, num_points, nodes_plot), linestyle='--', linewidth=1.5)
    if method == "spils_net":
        try:
            axs[4].plot(sm_results_internal, linestyle="--", linewidth=1.5, color=internal_state_color)
        except Exception as e:
            pass

# Add grids to all plots
for ax in axs.flat:
    ax.grid(True, linestyle=':', alpha=0.7)

x_positions = (np.arange(0, num_points/2, nodes_plot/2) * (100 / ((num_points/2)-1) ) - 50).astype(int)

# Legend Handles
location_handles = [
    mlines.Line2D([], [], color=colors_ordered[0], label=x_positions[0]),
    mlines.Line2D([], [], color=colors_ordered[1], label=x_positions[1]),
    mlines.Line2D([], [], color=colors_ordered[2], label=x_positions[2]),
    mlines.Line2D([], [], color=colors_ordered[3], label=x_positions[3]),
    mlines.Line2D([], [], color=colors_ordered[4], label=x_positions[4]),
    mlines.Line2D([], [], color=colors_ordered[5], label=x_positions[5]),
    mlines.Line2D([], [], color=colors_ordered[6], label=x_positions[6]),
]

if method == "spils_net":
    axs[4].legend(
        handles=[mlines.Line2D([], [], color=internal_state_color, label="Internal State")], 
        loc="best" # Matplotlib will automatically find a spot that doesn't overlap the line
    )

# 2. Handles for the data types (focusing on Linestyle & Thickness)
# Using 'k' (black) neutralizes the color, forcing the eye to look at the line style
style_handles = [
    mlines.Line2D([], [], color="k", label="Full\nFEM", linewidth=3, alpha=0.5),
    mlines.Line2D([], [], color="k", label=f"{method_legend_handles.get(method, method.upper())}\nPrediction", linestyle="--", linewidth=1.5),
]

# Add the Location legend to the top left
fig.legend(
    handles=location_handles,
    loc="upper left", 
    title="Interface Node x-Location [mm]", # Gives explicit context to the colors
    ncol=7, 
    bbox_to_anchor=(0.02, 0.99)
)

# Add the Style legend to the top right
fig.legend(
    handles=style_handles, 
    loc="upper right", 
    title="Model Output", # Gives explicit context to the linestyles
    ncol=2, 
    bbox_to_anchor=(0.98, 0.99)
)

# You may need to tweak the top rect value slightly depending on how tall the titles make the legends
if method == "spils_net":
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.94))
else:
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.93))

if freq == 740:
    section = {"benchmark": "04", "scaled": "05"}
else:
    section = {"benchmark": "99", "scaled": "99"}

plt.savefig(
    f"figures/{section[data_version]}_{method_legend_handles.get(method, method.upper()).lower()}_{data_version}_results_{freq}.pdf", 
    format="pdf", 
    bbox_inches="tight", 
    transparent=False,   # Keep it False for most journals to avoid display issues
    pad_inches=0.05      # Minimal whitespace around the edges
)

plt.show()

In [ ]:
show_predictions = True

method_legend_handles = {
    "spils_net": "SPILS-Net",
    "lstm": "LSTM",
}

num_points = tct_data_in.shape[1] // 3
if data_version == "benchmark":
    nodes_plot = 6
else:
    nodes_plot = 16

plt.rcParams.update({
    "font.family": "serif",   # Matches LaTeX default
    "font.size": 11,          # Standard paper font size
    "savefig.dpi": 300,       # Default save DPI
})

# 1. Change layout to 2 rows, 1 column. 
# Adjusted figsize to (10, 6) to keep proportions similar for 2 rows.
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Set y-labels and x-labels
axs[0].set_ylabel('Force x\n[N]')
axs[1].set_ylabel('Force y\n[N]')
axs[1].set_xlabel("Time step")

# 2. Limit the x-axis to only show time steps 3100 to 3200
# Because sharex=True, setting it on axs[0] applies to both.
axs[0].set_xlim(3180, 4000)

# Plot TRUE data for Forces (index 0 is Force X, index 1 is Force Y)
make_plot(axs[0], tct_data_out, range(0, num_points, nodes_plot), linewidth=3, alpha=0.5)
make_plot(axs[1], tct_data_out, range(1, num_points, nodes_plot), linewidth=3, alpha=0.5)

if show_predictions:
    # Plot PREDICTION data for Forces
    make_plot(axs[0], sm_results_out, range(0, num_points, nodes_plot), linestyle='--', linewidth=1.5)
    make_plot(axs[1], sm_results_out, range(1, num_points, nodes_plot), linestyle='--', linewidth=1.5)

axs[0].set_ylim(bottom=-5, top=5)
axs[1].set_ylim(bottom=610, top=850)  # Force y-axis to start at 0 for Force x

# Add grids to all plots
for ax in axs.flat:
    ax.grid(True, linestyle=':', alpha=0.7)

x_positions = (np.arange(0, num_points/2, nodes_plot/2) * (100 / ((num_points/2)-1) ) - 50).astype(int)

# Legend Handles
location_handles = [
    mlines.Line2D([], [], color=colors_ordered[0], label=x_positions[0]),
    mlines.Line2D([], [], color=colors_ordered[1], label=x_positions[1]),
    mlines.Line2D([], [], color=colors_ordered[2], label=x_positions[2]),
    mlines.Line2D([], [], color=colors_ordered[3], label=x_positions[3]),
    mlines.Line2D([], [], color=colors_ordered[4], label=x_positions[4]),
    mlines.Line2D([], [], color=colors_ordered[5], label=x_positions[5]),
    mlines.Line2D([], [], color=colors_ordered[6], label=x_positions[6]),
]

style_handles = [
    mlines.Line2D([], [], color="k", label="Full\nFEM", linewidth=3, alpha=0.5),
    mlines.Line2D([], [], color="k", label=f"{method_legend_handles.get(method, method.upper())}\nPrediction", linestyle="--", linewidth=1.5),
]

# Add the Location legend to the top left
fig.legend(
    handles=location_handles,
    loc="upper left", 
    title="Interface Node x-Location [mm]", 
    ncol=7, 
    # Adjusted Y anchor slightly for the smaller figure height
    bbox_to_anchor=(0.02, 1.05) 
)

# Add the Style legend to the top right
fig.legend(
    handles=style_handles, 
    loc="upper right", 
    title="Model Output", 
    ncol=2, 
    # Adjusted Y anchor slightly for the smaller figure height
    bbox_to_anchor=(0.98, 1.05)
)

# Adjusted top rect to leave room for the legends above the shorter figure
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.94))

section = {"benchmark": "04", "scaled": "05"}

# Added "_forces_zoom" to the filename to distinguish it from the full plot
plt.savefig(
    f"figures/{section[data_version]}_{method_legend_handles.get(method, method.upper()).lower()}_{data_version}_results_{freq}_forces_noise.pdf", 
    format="pdf", 
    bbox_inches="tight", 
    transparent=False,  
    pad_inches=0.05      
)

plt.show()

In [ ]:
# Load data
spils_error = np.load(f"results/sm_results_error_{data_version}_spils_net_v01_freq{freq}.npy")
lstm_error = np.load(f"results/sm_results_error_{data_version}_lstm_v01_freq{freq}.npy")

time_steps = spils_error.shape[0]

plt.rcParams.update({
    "font.family": "serif",   # Matches LaTeX default
    "font.size": 11,          # Standard paper font size
    "savefig.dpi": 300,       # Default save DPI
})

fig, ax = plt.subplots(figsize=(10, 5))

# Plot raw lines
# Using a slightly thinner linewidth (1.0) helps prevent the plot 
# from looking like a solid block of color if the frequency is high.
ax.plot(spils_error, color="tab:blue", label="SPILS Net", linewidth=1.5)
ax.plot(lstm_error, color="tab:orange", label="LSTM", linewidth=1.5)

# Axis Styling
ax.set_xlabel("Time step")
ax.set_ylabel("Error $e_n$")
ax.set_xlim(0, time_steps)

# Set Y-limit based on the actual data peak + a small margin
y_max = np.max(np.concatenate((spils_error, lstm_error))) * 1.05
ax.set_ylim(0, y_max)

# Add a light grid for better readability of values
ax.grid(True, linestyle=':', alpha=0.6)

# Legend
ax.legend(loc="upper left", frameon=True, shadow=False)

fig.tight_layout()

plt.savefig(
    f"figures/{section[data_version]}_{data_version}_error.pdf", 
    format="pdf", 
    bbox_inches="tight", 
    transparent=False,   # Keep it False for most journals to avoid display issues
    pad_inches=0.05      # Minimal whitespace around the edges
)

plt.show()

In [ ]:
freqs = [740, 1320, 1880]
# Define styles: one for each frequency
line_styles = {740: '-', 1320: '--', 1880: ':'}
# Define colors: one for each method
colors = {'SPILS': 'tab:blue', 'LSTM': 'tab:orange'}

plt.rcParams.update({
    "font.family": "serif",   # Matches LaTeX default
    "font.size": 11,          # Standard paper font size
    "savefig.dpi": 300,       # Default save DPI
})

plt.figure(figsize=(10, 6))

all_data = [] # To track max y-limit

for fr in freqs:
    # Load data
    with open(f"results/sm_results_error_{data_version}_lstm_v01_freq{fr}.npy", "rb") as f:
        lstm_error = np.load(f)
    with open(f"results/sm_results_error_{data_version}_spils_net_v01_freq{fr}.npy", "rb") as f:
        spils_error = np.load(f)
    
    all_data.extend([lstm_error, spils_error])
    
    # Plot curves with specific color (method) and linestyle (frequency)
    plt.plot(spils_error, color=colors['SPILS'], linestyle=line_styles[fr], alpha=0.8)
    plt.plot(lstm_error, color=colors['LSTM'], linestyle=line_styles[fr], alpha=0.8)

    # Optional: Calculate and plot means for each if desired
    spils_mean = np.mean(spils_error)
    lstm_mean = np.mean(lstm_error)

# Formatting
y_lim = np.max(all_data) * 1.1
plt.ylim(0, y_lim)
plt.xlabel("Time step")
plt.ylabel("$e_n$")
plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

# --- Custom Legend ---
# This creates a split legend: Methods (Colors) and Frequencies (Line Styles)
method_lines = [
    mlines.Line2D([], [], color=colors['SPILS'], label='SPILS Net'),
    mlines.Line2D([], [], color=colors['LSTM'], label='LSTM')
]

freq_lines = [
    mlines.Line2D([], [], color='gray', linestyle=line_styles[740], label='740 Hz'),
    mlines.Line2D([], [], color='gray', linestyle=line_styles[1320], label='1320 Hz'),
    mlines.Line2D([], [], color='gray', linestyle=line_styles[1880], label='1880 Hz')
]

# Combine both lists for the legend
plt.legend(handles=freq_lines + method_lines, loc='upper left', ncol=2)

plt.tight_layout()

plt.savefig(
    "figures/05_frequency_analysis.pdf", 
    format="pdf", 
    bbox_inches="tight", 
    transparent=False,   # Keep it False for most journals to avoid display issues
    pad_inches=0.05      # Minimal whitespace around the edges
)

plt.show()